In [1]:
import pandas as pd
import itertools

In [2]:
def get_num_of_potential_options(num_matchups, outcomes=1):
    
    # Round:         Sweet 16 
    # Num. Matchups: 8
    # Math:          (2**8)*(2**4)*(2**2)*(2**1)
    
    outcomes*=(2**num_matchups)
    if num_matchups == 1:
        return int(outcomes)
    else:
        return get_num_of_potential_options(num_matchups/2, outcomes)

In [3]:
get_num_of_potential_options(1)

2

In [4]:
2 147 483 648

SyntaxError: invalid syntax (2441789518.py, line 1)

In [3]:
def get_inprogress_options(actual_df, current_round):
    
    # Returns all options for completed rounds and for the current round
    # Current round can be either unstarted or inprogress (some games complete)
    # Note: for completed rounds there is only one outcome option since
    # the contests have already been completed 
    
    field = actual_df[current_round].dropna().to_list()
#     field = actual_df[current_round].to_list()
    
    next_round = {
        'Round of 64': 'Round of 32', 
        'Round of 32': 'Sweet 16',
        'Sweet 16': 'Elite 8',
        'Elite 8': 'Final 4',
        'Final 4': 'Championship',
        'Championship': 'Winner',
    }[current_round]

    chunk_size = {
        'Round of 64': 2,
        'Round of 32': 4,
        'Sweet 16': 8,
        'Elite 8': 16,
        'Final 4': 32,
        'Championship': 64,
    }[current_round]
    
    winners = set(actual_df[next_round].dropna().to_list())
    eliminated_teams = set(field).difference(winners)
    
    completed_contests = len(winners)
    uncomplete_contests = int((len(field)/2) - completed_contests)
    total_contests = completed_contests + uncomplete_contests
    potential_outcomes = max([1,2**uncomplete_contests])
    
    
    print(f'{current_round}: {completed_contests} completed contests out of {total_contests}')
    print(f'Num. Potential Outcomes = {potential_outcomes}')
    print(f'-----------------')
    
    match_ups = [] 
    for i in range(0, 64, chunk_size):
        match_up = set(actual_df[current_round][i:i + chunk_size].dropna().to_list())
        intersection = match_up.intersection(winners)
        if intersection:
            match_ups.append(intersection)
        else:
            match_ups.append(match_up)
    all_combinations = list(itertools.product(*match_ups))
    return all_combinations


def get_round_options(actual_df, teams):
    
    # Input: list of teams in the round
    # Output: all possible outcomes from those contests
    
    chunk_size_dict = {
        0: 2,
        32: 4,
        48: 8,
        56: 16,
        60: 32,
        62: 64
    }
    
    field = actual_df['Round of 64'].to_list()
    eliminated_teams = set(field).difference(set(teams))
    chunk_size = chunk_size_dict[len(eliminated_teams)]
    match_ups = [] 
    for i in range(0, 64, chunk_size):
        match_up = set(field[i:i + chunk_size]).difference(eliminated_teams)
        match_ups.append(match_up)
    options = list(itertools.product(*match_ups))
    
    return options

In [5]:
FILE_PATH = 'MWI 2026 Auto v2.xlsx'
actual_df = pd.read_excel(FILE_PATH, sheet_name='Actual Results')

### Note: There are too many options to run this until the round of 32 has at least been partially decided
With 32 teams left, there are 16 matchups in that round and 2,147,483,648 potential final outcomes  
With 16 teams left, there are 8 matchups in that round and 32,768 potential final outcomes  
With 8 teams left, there are 4 matchups in that round and 128 potential final outcomes  
With 4 teams left, there are 2 matchups in that round and 8 potential final outcomes  
With 2 teams left, there are 1 matchups in that round and 1 potential final outcomes

## _Partially_ Completed Round of 32 to Champion Options
With 32 teams, and all 16 matchups incomplete there are 2,147,483,648 potential final outcomes. It takes a very long time to calculate all of these outcomes, so it's best to not run this until some of the matchups have been completed.

In [13]:
all_paths = []

# complete and in-progress rounds
for round_64_option in get_inprogress_options(actual_df, 'Round of 64'):
    for round_32_option in get_inprogress_options(actual_df, 'Round of 32'):
        # future rounds
        for sweet_16_option in get_round_options(actual_df, round_32_option):
            for elite_8_option in get_round_options(actual_df, sweet_16_option):
                for final_4_option in get_round_options(actual_df, elite_8_option):
                    for championship_option in get_round_options(actual_df, final_4_option):
                        all_paths.append([
                            round_64_option, 
                            round_32_option, 
                            sweet_16_option, 
                            elite_8_option, 
                            final_4_option, 
                            championship_option
                        ])
print('')
print(f'Num. of Potential Final Outcomes: {len(all_paths)}')  


Num. of Potential Final Outcomes: 32768


## Sweet 16 to Champion Options

In [6]:
all_paths = []

# complete and in-progress rounds
for round_64_option in get_inprogress_options(actual_df, 'Round of 64'):
    for round_32_option in get_inprogress_options(actual_df, 'Round of 32'):
        # future rounds
        for sweet_16_option in get_round_options(actual_df, round_32_option):
            for elite_8_option in get_round_options(actual_df, sweet_16_option):
                for final_4_option in get_round_options(actual_df, elite_8_option):
                    for championship_option in get_round_options(actual_df, final_4_option):
                        all_paths.append([
                            round_64_option, 
                            round_32_option, 
                            sweet_16_option, 
                            elite_8_option, 
                            final_4_option, 
                            championship_option
                        ])
print('')
print(f'Num. of Potential Final Outcomes: {len(all_paths)}')     

Round of 64: 32 completed contests out of 32
Num. Potential Outcomes = 1
-----------------
Round of 32: 16 completed contests out of 16
Num. Potential Outcomes = 1
-----------------

Num. of Potential Final Outcomes: 32768


## Elite 8 to Champion Options

In [15]:
all_paths = []

# complete and in-progress rounds
for round_64_option in get_inprogress_options(actual_df, 'Round of 64'):
    for round_32_option in get_inprogress_options(actual_df, 'Round of 32'):
        for sweet_16_option in get_inprogress_options(actual_df, 'Sweet 16'):
            # future rounds
            for elite_8_option in get_round_options(actual_df, sweet_16_option):
                for final_4_option in get_round_options(actual_df, elite_8_option):
                    for championship_option in get_round_options(actual_df, final_4_option):
                        all_paths.append([
                            round_64_option, 
                            round_32_option, 
                            sweet_16_option, 
                            elite_8_option, 
                            final_4_option, 
                            championship_option
                        ])
print('')
print(f'Num. of Potential Final Outcomes: {len(all_paths)}')     


Num. of Potential Final Outcomes: 128


## Final 4 to Champion Options

In [10]:
all_paths = []

# complete and in-progress rounds
for round_64_option in get_inprogress_options(actual_df, 'Round of 64'):
    for round_32_option in get_inprogress_options(actual_df, 'Round of 32'):
        for sweet_16_option in get_inprogress_options(actual_df, 'Sweet 16'):
            for elite_8_option in get_inprogress_options(actual_df, 'Elite 8'):
                # future rounds
                for final_4_option in get_round_options(actual_df, elite_8_option):
                    for championship_option in get_round_options(actual_df, final_4_option):
                        all_paths.append([
                            round_64_option, 
                            round_32_option, 
                            sweet_16_option, 
                            elite_8_option, 
                            final_4_option, 
                            championship_option
                        ])
print('')
print(f'Num. of Potential Final Outcomes: {len(all_paths)}')  

Round of 64: 32 completed contests out of 32
Num. Potential Outcomes = 1
-----------------
Round of 32: 16 completed contests out of 16
Num. Potential Outcomes = 1
-----------------
Sweet 16: 4 completed contests out of 8
Num. Potential Outcomes = 16
-----------------
Elite 8: 0 completed contests out of 2
Num. Potential Outcomes = 4
-----------------
Elite 8: 0 completed contests out of 2
Num. Potential Outcomes = 4
-----------------
Elite 8: 0 completed contests out of 2
Num. Potential Outcomes = 4
-----------------
Elite 8: 0 completed contests out of 2
Num. Potential Outcomes = 4
-----------------
Elite 8: 0 completed contests out of 2
Num. Potential Outcomes = 4
-----------------
Elite 8: 0 completed contests out of 2
Num. Potential Outcomes = 4
-----------------
Elite 8: 0 completed contests out of 2
Num. Potential Outcomes = 4
-----------------
Elite 8: 0 completed contests out of 2
Num. Potential Outcomes = 4
-----------------
Elite 8: 0 completed contests out of 2
Num. Potenti

## Championship to Champion Options

In [113]:
all_paths = []

# complete and in-progress rounds
for round_64_option in get_inprogress_options(actual_df, 'Round of 64'):
    for round_32_option in get_inprogress_options(actual_df, 'Round of 32'):
        for sweet_16_option in get_inprogress_options(actual_df, 'Sweet 16'):
            for elite_8_option in get_inprogress_options(actual_df, 'Elite 8'):
                for final_4_option in get_inprogress_options(actual_df, 'Final 4'):
                    for championship_option in get_inprogress_options(actual_df, 'Championship'):
                        all_paths.append([
                            round_64_option, 
                            round_32_option, 
                            sweet_16_option, 
                            elite_8_option, 
                            final_4_option, 
                            championship_option
                        ])
print('')
print(f'Num. of Potential Final Outcomes: {len(all_paths)}') 

Round of 64: 32 completed contests out of 32
Num. Potential Outcomes = 1
-----------------
Round of 32: 16 completed contests out of 16
Num. Potential Outcomes = 1
-----------------
Sweet 16: 8 completed contests out of 8
Num. Potential Outcomes = 1
-----------------
Elite 8: 4 completed contests out of 4
Num. Potential Outcomes = 1
-----------------
Final 4: 2 completed contests out of 2
Num. Potential Outcomes = 1
-----------------
Championship: 0 completed contests out of 1
Num. Potential Outcomes = 2
-----------------

Num. of Potential Final Outcomes: 2


In [18]:
FILE_PATH = 'MWI 2026 Auto v2.xlsx'

# copied from mwi.leaderboard['MWI Contestant'].tolist()
CONTESTANTS = [
    'Truett',
     'Eli',
     'Don',
     'Tim',
     'Roman',
     'Penny',
     'Arden',
     'Chad',
     'Elaine',
     'Sarah',
     'Maren',
     'Cynthia',
     'Emma',
     'August',
     'Henny',
     'Colombina',
     'Shawn',
     'Indie',
     'Mary',
     'Lucia',
     'Pete',
     'Sabine',
     'Winter',
     'Pia',
     'Chalamet']

ROUNDS = ['Round of 32', 'Sweet 16', 'Elite 8', 'Final 4', 'Championship', 'Winner']

In [19]:
brackets_dict = pd.read_excel(FILE_PATH, sheet_name=None)

In [20]:
brackets_dict['Shawn']['Sweet 16'].dropna().to_list()

['(1) Duke',
 '(4) Kansas',
 '(3) Michigan St',
 '(2) UConn',
 '(1) Florida',
 '(5) Vanderbilt',
 '(6) N. Carolina',
 '(2) Houston',
 '(1) Arizona',
 '(5) Wisconsin',
 '(3) Gonzaga',
 '(2) Purdue',
 '(1) Michigan',
 '(5) Texas Tech',
 '(6) Tennessee',
 '(2) Iowa St']

In [52]:
remainings_teams = ['(1) Uconn', '(1) North Carolina', '(1) Purdue', '(1) Houston',
                    '(2) Iowa St.', '(2) Arizona', '(2) Marquette', '(2) Tennessee']

In [13]:
field = brackets_dict['Actual Results']['Round of 64'].to_list()
eliminated_teams = set(field).difference(set(remainings_teams))
len(eliminated_teams)

NameError: name 'remainings_teams' is not defined

In [21]:
def get_chunk_size(input_value):
    
    chunk_size_dict = {
        range(0, 32): 2,
        range(32, 48): 4,
        range(48, 56) : 8,
        range(56, 60) : 16,
        range(60, 62): 32,
        range(62, 64): 64
    }
    
    for key_range, value in chunk_size_dict.items():
        if input_value in key_range:
            return value
    # Return None or an appropriate value if input_value doesn't fall within any range
    return None

get_chunk_size(64-23)

4

In [22]:
def dev_get_round_options(teams):
    
    field = brackets_dict['Actual Results']['Round of 64'].to_list()
    eliminated_teams = set(field).difference(set(teams))
    chunk_size = get_chunk_size(len(eliminated_teams))
    match_ups = [] 
    for i in range(0, 64, chunk_size):
        match_up = set(field[i:i + chunk_size]).difference(eliminated_teams)
        match_ups.append(match_up)
    options = list(itertools.product(*match_ups))
    
    return options

In [23]:
def get_remaining_teams(current_round, actual_df):
    
    remaining_teams = set(actual_df[current_round].dropna().to_list())
    
    chunk_size = {
        'Round of 32': 2,
        'Sweet 16': 4,
        'Elite 8': 8,
        'Final 4': 16,
        'Championship': 32,
        'Winner': 64,
    }[current_round]
    
    previous_round = {
        'Round of 32': 'Round of 64',
        'Sweet 16': 'Round of 32',
        'Elite 8': 'Sweet 16',
        'Final 4': 'Elite 8',
        'Championship': 'Final 4',
        'Winner': 'Championship',
        'Champion': 'Winner',
    }[current_round]
    
    field = set(actual_df[previous_round].dropna().to_list())
    winners = set(actual_df[current_round].dropna().to_list())
    
    match_ups = []
    for i in range(0, 64, chunk_size):
        match_ups.append(actual_df[i:i + chunk_size][previous_round].dropna().to_list())
    
    for match_up in match_ups:
        if remaining_teams.intersection(set(match_up)):
            pass
        else:
            remaining_teams.update(set(match_up))
    
    return remaining_teams

get_remaining_teams('Sweet 16', brackets_dict['Actual Results'])

{'(1) Arizona',
 '(1) Duke',
 '(1) Michigan',
 '(11) Texas',
 '(2) Houston',
 '(2) Iowa St',
 '(2) Purdue',
 '(2) UConn',
 '(3) Illinois',
 '(3) Michigan St',
 '(4) Alabama',
 '(4) Arkansas',
 '(4) Nebraska',
 "(5) St. John's",
 '(6) Tennessee',
 '(9) Iowa'}

In [14]:
set([1,2]).union(set([3,4]))

{1, 2, 3, 4}

In [92]:
remaining_teams = set(brackets_dict['Actual Results'][''].to_list()).difference(set(['(1) Houston', '(1) North Carolina']))

In [ ]:
dev_get_round_options(remaining_teams)

In [27]:
for round_64_option in get_inprogress_options(actual_df, 'Round of 64'):
    print(round_64_option)

('(1) Auburn', '(9) Creighton', '(5) Michigan', '(4) Texas A&M', '(6) Ole Miss', '(3) Iowa State', '(10) New Mexico', '(2) Michigan St', '(1) Florida', '(8) UConn', '(12) Colorado St', '(4) Maryland', '(11) Drake', '(3) Texas Tech', '(10) Arkansas', "(2) St John's", '(1) Duke', '(9) Baylor', '(5) Oregon', '(4) Arizona', '(6) BYU', '(3) Wisconsin', "(7) Saint Mary's", '(2) Alabama', '(1) Houston', '(8) Gonzaga', '(12) McNeese', '(4) Purdue', '(6) Illinois', '(3) Kentucky', '(7) UCLA', '(2) Tennessee')


In [35]:
actual_df = pd.read_excel('MWI 2026 Auto v2.xlsx', sheet_name='Actual Results')

get_inprogress_options(actual_df, 'Sweet 16')

[("(5) St. John's",
  '(3) Michigan St',
  '(4) Nebraska',
  '(3) Illinois',
  '(1) Arizona',
  '(2) Purdue',
  '(4) Alabama',
  '(6) Tennessee'),
 ("(5) St. John's",
  '(3) Michigan St',
  '(4) Nebraska',
  '(3) Illinois',
  '(1) Arizona',
  '(2) Purdue',
  '(4) Alabama',
  '(2) Iowa St'),
 ("(5) St. John's",
  '(3) Michigan St',
  '(4) Nebraska',
  '(3) Illinois',
  '(1) Arizona',
  '(2) Purdue',
  '(1) Michigan',
  '(6) Tennessee'),
 ("(5) St. John's",
  '(3) Michigan St',
  '(4) Nebraska',
  '(3) Illinois',
  '(1) Arizona',
  '(2) Purdue',
  '(1) Michigan',
  '(2) Iowa St'),
 ("(5) St. John's",
  '(3) Michigan St',
  '(4) Nebraska',
  '(3) Illinois',
  '(1) Arizona',
  '(11) Texas',
  '(4) Alabama',
  '(6) Tennessee'),
 ("(5) St. John's",
  '(3) Michigan St',
  '(4) Nebraska',
  '(3) Illinois',
  '(1) Arizona',
  '(11) Texas',
  '(4) Alabama',
  '(2) Iowa St'),
 ("(5) St. John's",
  '(3) Michigan St',
  '(4) Nebraska',
  '(3) Illinois',
  '(1) Arizona',
  '(11) Texas',
  '(1) Michi

In [36]:
def get_inprogress_options(actual_df, current_round):
    
    field = actual_df[current_round].to_list()
    
    next_round = {
        'Round of 64': 'Round of 32', 
        'Round of 32': 'Sweet 16',
        'Sweet 16': 'Elite 8',
        'Elite 8': 'Final 4',
        'Final 4': 'Championship',
        'Championship': 'Winner',
    }[current_round]

    chunk_size = {
        'Round of 64': 2,
        'Round of 32': 4,
        'Sweet 16': 8,
        'Elite 8': 16,
        'Final 4': 32,
        'Championship': 64,
    }[current_round]
    
    winners = set(actual_df[next_round].dropna().to_list())
    eliminated_teams = set(field).difference(winners)
    
    match_ups = [] 
    for i in range(0, 64, chunk_size):
        match_up = set(actual_df[current_round][i:i + chunk_size].dropna().to_list())
        intersection = match_up.intersection(winners)
        if intersection:
            match_ups.append(intersection)
        else:
            match_ups.append(match_up)
    all_combinations = list(itertools.product(*match_ups))
    return all_combinations


def get_round_options(actual_df, teams):
    
    chunk_size_dict = {
        0: 2,
        32: 4,
        48: 8,
        56: 16,
        60: 32,
        62: 64
    }
    
    field = actual_df['Round of 64'].to_list()
    eliminated_teams = set(field).difference(set(teams))
    chunk_size = chunk_size_dict[len(eliminated_teams)]
    match_ups = [] 
    for i in range(0, 64, chunk_size):
        match_up = set(field[i:i + chunk_size]).difference(eliminated_teams)
        match_ups.append(match_up)
    options = list(itertools.product(*match_ups))
    
    return options

actual_df = pd.read_excel('MWI 2026 Auto v2.xlsx', sheet_name='Actual Results')

round_64_options = get_inprogress_options(actual_df, 'Round of 64')
round_32_options = get_inprogress_options(actual_df, 'Round of 32')
sweet_16_options = get_inprogress_options(actual_df, 'Sweet 16')
elite_8_options = get_inprogress_options(actual_df, 'Elite 8')
final_4_options = get_inprogress_options(actual_df, 'Final 4')
championship_options = get_inprogress_options(actual_df, 'Championship')

all_paths = []

# *** OPTIONS MATH ***
# *** use this structure when future rounds exist***
# complete and in-progress rounds
for round_64_option in get_inprogress_options(actual_df, 'Round of 64'):
    for round_32_option in get_inprogress_options(actual_df, 'Round of 32'):
        
        # future rounds
        for sweet_16_option in get_inprogress_options(actual_df, 'Sweet 16'):
            
            for elite_8_option in get_inprogress_options(actual_df, sweet_16_option):
                
                for final_4_option in get_inprogress_options(actual_df, elite_8_option):
                    
                    championship_options = get_round_options(actual_df, final_4_option)
                    for championship_option in championship_options:
                        all_paths.append([
                            round_64_option, 
                            round_32_option, 
                            sweet_16_option, 
                            elite_8_option, 
                            final_4_option, 
                            championship_option
                        ])
                        
# # *** OPTIONS MATH ***
# # *** use this structure when future rounds exist***
# # complete and in-progress rounds
# for round_64_option in get_inprogress_options(actual_df, 'Round of 64'):
#     for round_32_option in get_inprogress_options(actual_df, 'Round of 32'):
#         for sweet_16_option in get_inprogress_options(actual_df, 'Sweet 16'):
            
#             for elite_8_option in get_inprogress_options(actual_df, 'Elite 8'):
#                 # future rounds
#                 for final_4_option in get_inprogress_options(actual_df, 'Final 4'):
                    
#                     championship_options = get_round_options(actual_df, final_4_option)
#                     for championship_option in championship_options:
#                         all_paths.append([
#                             round_64_option, 
#                             round_32_option, 
#                             sweet_16_option, 
#                             elite_8_option, 
#                             final_4_option, 
#                             championship_option
#                         ])

# # complete and in-progress rounds
# for round_64_option in get_inprogress_options(actual_df, 'Round of 64'):
# #     print(round_64_option)
#     for round_32_option in get_inprogress_options(actual_df, 'Round of 32'):
# #         print(round_32_option)
#         for sweet_16_option in get_inprogress_options(actual_df, 'Sweet 16'):
# #             print(sweet_16_option)
#             for elite_8_option in get_inprogress_options(actual_df, 'Elite 8'):
#                 print(elite_8_option)
#                 for final_4_option in get_inprogress_options(actual_df, 'Final 4'):
#                     for championship_option in get_inprogress_options(actual_df, 'Championship'):
#                         all_paths.append([
#                             round_64_option, 
#                             round_32_option, 
#                             sweet_16_option, 
#                             elite_8_option, 
#                             final_4_option, 
#                             championship_option
#                         ])
len(all_paths)

KeyError: ("(5) St. John's", '(3) Michigan St', '(4) Nebraska', '(3) Illinois', '(1) Arizona', '(2) Purdue', '(4) Alabama', '(6) Tennessee')

In [33]:
all_paths[0]

[('(1) Auburn',
  '(9) Creighton',
  '(5) Michigan',
  '(4) Texas A&M',
  '(6) Ole Miss',
  '(3) Iowa State',
  '(10) New Mexico',
  '(2) Michigan St',
  '(1) Florida',
  '(8) UConn',
  '(12) Colorado St',
  '(4) Maryland',
  '(11) Drake',
  '(3) Texas Tech',
  '(10) Arkansas',
  "(2) St John's",
  '(1) Duke',
  '(9) Baylor',
  '(5) Oregon',
  '(4) Arizona',
  '(6) BYU',
  '(3) Wisconsin',
  "(7) Saint Mary's",
  '(2) Alabama',
  '(1) Houston',
  '(8) Gonzaga',
  '(12) McNeese',
  '(4) Purdue',
  '(6) Illinois',
  '(3) Kentucky',
  '(7) UCLA',
  '(2) Tennessee'),
 ('(1) Auburn',
  '(5) Michigan',
  '(6) Ole Miss',
  '(2) Michigan St',
  '(1) Florida',
  '(4) Maryland',
  '(3) Texas Tech',
  '(10) Arkansas',
  '(1) Duke',
  '(4) Arizona',
  '(6) BYU',
  '(2) Alabama',
  '(1) Houston',
  '(4) Purdue',
  '(3) Kentucky',
  '(2) Tennessee'),
 ('(1) Auburn',
  '(2) Michigan St',
  '(1) Florida',
  '(3) Texas Tech',
  '(1) Duke',
  '(2) Alabama',
  '(1) Houston',
  '(2) Tennessee'),
 ('(1) Au

In [34]:
all_paths_sets = [[set(j) for j in result] for result in all_paths]

In [35]:
brackets = pd.read_excel('MWI 2025 Auto v2.xlsx', sheet_name=None)

def get_contestant_sets(brackets):
    
    contestants_dict = {}
    for sheet_name, contestant_df in brackets.items():
        if sheet_name != 'Actual Results' and 'Contestant' not in sheet_name:
            contestants_dict[sheet_name] = []
            for col in contestant_df.columns: 
                if col != 'Round of 64':
                    contestants_dict[sheet_name].append(set(contestant_df[col].dropna().to_list()))

    return contestants_dict  

contestant_sets = get_contestant_sets(brackets)
len(contestant_sets)

24

In [36]:
def score_bracket(contestant_sets, hypothetical_sets, print_round_results=False):
    
    index_round_names = {
        0: 'round of 64',
        1: 'round of 32',
        2: 'sweet 16',
        3: 'elite 8',
        4: 'final 4',
        5: 'final',
    }
    
    index_round_points = {
        0: 10,    # round of 64
        1: 20,    # round of 32
        2: 40,    # sweet 16
        3: 80,    # elite 8
        4: 160,   # final 4
        5: 320,   # final
    }
    
    final_score = 0
    round_results = [len(s1&s2) for s1,s2 in zip(contestant_sets, hypothetical_sets)]
    for index, correct in enumerate(round_results):
        if print_round_results:
            print(f'{index_round_names[index]} = {correct * index_round_points[index]}')
        final_score += (correct * index_round_points[index])
    
    return final_score

In [37]:
def get_contestants_best_path(contestant, all_paths_sets):
    max_score = 0
    best_path = None

    for path in all_paths_sets:
        _score = score_bracket(contestant_sets[contestant], path)
        if _score > max_score:
            max_score = _score
            best_path = path
    print(max_score)
    
    return best_path

get_contestants_best_path('Maren', all_paths_sets)

500


[{'(1) Auburn',
  '(1) Duke',
  '(1) Florida',
  '(1) Houston',
  '(10) Arkansas',
  '(10) New Mexico',
  '(11) Drake',
  '(12) Colorado St',
  '(12) McNeese',
  '(2) Alabama',
  '(2) Michigan St',
  "(2) St John's",
  '(2) Tennessee',
  '(3) Iowa State',
  '(3) Kentucky',
  '(3) Texas Tech',
  '(3) Wisconsin',
  '(4) Arizona',
  '(4) Maryland',
  '(4) Purdue',
  '(4) Texas A&M',
  '(5) Michigan',
  '(5) Oregon',
  '(6) BYU',
  '(6) Illinois',
  '(6) Ole Miss',
  "(7) Saint Mary's",
  '(7) UCLA',
  '(8) Gonzaga',
  '(8) UConn',
  '(9) Baylor',
  '(9) Creighton'},
 {'(1) Auburn',
  '(1) Duke',
  '(1) Florida',
  '(1) Houston',
  '(10) Arkansas',
  '(2) Alabama',
  '(2) Michigan St',
  '(2) Tennessee',
  '(3) Kentucky',
  '(3) Texas Tech',
  '(4) Arizona',
  '(4) Maryland',
  '(4) Purdue',
  '(5) Michigan',
  '(6) BYU',
  '(6) Ole Miss'},
 {'(1) Auburn',
  '(1) Duke',
  '(1) Florida',
  '(1) Houston',
  '(2) Alabama',
  '(2) Michigan St',
  '(2) Tennessee',
  '(3) Texas Tech'},
 {'(1) Au

In [38]:
max_score = 0
best_path = None

for path in all_paths_sets:
    _score = score_bracket(contestant_sets['Maren'], path)
    if _score > max_score:
        max_score = _score
        best_path = path
        


In [39]:
score_bracket(contestant_sets['Shawn'], best_path, True)

round of 64 = 240
round of 32 = 240
sweet 16 = 280
elite 8 = 240
final 4 = 0
final = 0


1000

In [40]:
(320*5)+260

1860

In [41]:
len(contestant_sets['Shawn'][0])

32

In [42]:
for contestant, bracket in contestant_sets.items():
    print(contestant)
    _score = score_bracket(bracket, best_path)
    print(_score)
    print('----------')

Shawn
1000
----------
Maren
500
----------
Elaine
1460
----------
Colombina
1210
----------
Roman
1150
----------
Eli
1240
----------
Pete
560
----------
Letty
1530
----------
Ofelia
1060
----------
Sarah
770
----------
DOGE
330
----------
Cynthia
920
----------
Tim
1170
----------
Emma
1370
----------
Mary
850
----------
Chad
760
----------
Truett
1190
----------
DOAC
970
----------
Sabine
420
----------
Pia
310
----------
August
340
----------
Penny
1410
----------
Don
1060
----------
Indie
970
----------


In [43]:
best_path

[{'(1) Auburn',
  '(1) Duke',
  '(1) Florida',
  '(1) Houston',
  '(10) Arkansas',
  '(10) New Mexico',
  '(11) Drake',
  '(12) Colorado St',
  '(12) McNeese',
  '(2) Alabama',
  '(2) Michigan St',
  "(2) St John's",
  '(2) Tennessee',
  '(3) Iowa State',
  '(3) Kentucky',
  '(3) Texas Tech',
  '(3) Wisconsin',
  '(4) Arizona',
  '(4) Maryland',
  '(4) Purdue',
  '(4) Texas A&M',
  '(5) Michigan',
  '(5) Oregon',
  '(6) BYU',
  '(6) Illinois',
  '(6) Ole Miss',
  "(7) Saint Mary's",
  '(7) UCLA',
  '(8) Gonzaga',
  '(8) UConn',
  '(9) Baylor',
  '(9) Creighton'},
 {'(1) Auburn',
  '(1) Duke',
  '(1) Florida',
  '(1) Houston',
  '(10) Arkansas',
  '(2) Alabama',
  '(2) Michigan St',
  '(2) Tennessee',
  '(3) Kentucky',
  '(3) Texas Tech',
  '(4) Arizona',
  '(4) Maryland',
  '(4) Purdue',
  '(5) Michigan',
  '(6) BYU',
  '(6) Ole Miss'},
 {'(1) Auburn',
  '(1) Duke',
  '(1) Florida',
  '(1) Houston',
  '(2) Alabama',
  '(2) Michigan St',
  '(2) Tennessee',
  '(3) Texas Tech'},
 {'(1) Au

In [44]:
final_results_dict = {contestant:0 for contestant in contestant_sets.keys()}
cynthia_options = []

for index, hypo_set in enumerate(all_paths_sets):
    if index % 100000 == 0:
        print(index)
    results = []
    for contestant, contestant_set in contestant_sets.items():
        brackets[contestant] = contestant_set
        score = score_bracket(contestant_set, hypo_set)
        results.append((score, contestant))
    _max = max(results)[0]
    for score, contestant in results:
        if score == _max:
            final_results_dict[contestant] +=1
            if contestant == 'Cynthia':
                cynthia_options.append(hypo_set)

0


In [55]:
for index, hypo_set in enumerate(all_paths_sets):
    print(hypo_set)
    if index % 100000 == 0:
        print(index)
    results = []
    for contestant, contestant_set in contestant_sets.items():
        brackets[contestant] = contestant_set
        score = score_bracket(contestant_set, hypo_set)
        results.append((score, contestant))
    _max = max(results)[0]
    for score, contestant in results:
        if score == _max:
            print(contestant)
            final_results_dict[contestant] +=1
            if contestant == 'Cynthia':
                cynthia_options.append(hypo_set)
    print('----')

[{'(8) UConn', '(4) Arizona', '(6) Illinois', '(10) New Mexico', '(12) McNeese', '(2) Tennessee', "(7) Saint Mary's", '(3) Kentucky', '(1) Duke', '(1) Auburn', '(1) Houston', '(6) Ole Miss', '(4) Maryland', '(12) Colorado St', '(3) Texas Tech', '(9) Creighton', '(5) Oregon', '(4) Purdue', '(3) Wisconsin', '(8) Gonzaga', '(7) UCLA', '(6) BYU', '(3) Iowa State', '(2) Alabama', '(4) Texas A&M', '(5) Michigan', '(9) Baylor', '(2) Michigan St', "(2) St John's", '(11) Drake', '(10) Arkansas', '(1) Florida'}, {'(5) Michigan', '(6) Ole Miss', '(4) Maryland', '(3) Texas Tech', '(4) Purdue', '(2) Michigan St', '(4) Arizona', '(10) Arkansas', '(6) BYU', '(2) Tennessee', '(3) Kentucky', '(1) Duke', '(1) Auburn', '(1) Houston', '(2) Alabama', '(1) Florida'}, {'(3) Texas Tech', '(2) Michigan St', '(2) Tennessee', '(1) Duke', '(1) Auburn', '(1) Houston', '(2) Alabama', '(1) Florida'}, {'(1) Duke', '(1) Auburn', '(1) Houston', '(1) Florida'}, {'(1) Duke', '(1) Auburn'}, {'(1) Duke'}]
0
Letty
----
[{'(

In [ ]:
Tim

In [51]:
ways_df = pd.DataFrame({'Contestant': final_results_dict.keys(),
              'Ways to Win': final_results_dict.values()}).sort_values(by='Ways to Win', ascending=False)


for i in ways_df[ways_df['Ways to Win']>0]['Contestant'].tolist():
    print(i)
    
ways_df

Tim
Letty
Penny
Indie
DOAC


,Contestant,Ways to Win
12,Tim,2
7,Letty,2
21,Penny,2
23,Indie,2
17,DOAC,1
4,Roman,0
5,Eli,0
22,Don,0
2,Elaine,0
20,August,0


In [315]:
t = [[o[3:6]] for o in cynthia_options]

for _ in t:
    for _y in _:
        for _i in _y:
            print(_i)
    print('------')

{'(4) Alabama', '(11) NC State', '(1) Purdue', '(1) Uconn'}
{'(4) Alabama', '(1) Purdue'}
{'(1) Purdue'}
------


In [177]:
import dataframe_image as dfi
(ways_df
     .style.hide_index()
     .set_caption('    ')
     .set_table_styles([{
            'selector': 'caption',
            'props': [
                ('color', 'black'),
                ('font-size', '20px')
            ]
        }])
    .export_png('Ways to Win.png', dpi=600)
)

[0324/204643.391414:WARNING:runtime_features.cc(728)] AttributionReportingCrossAppWeb cannot be enabled in this configuration. Use --enable-features=ConversionMeasurement,AttributionReportingCrossAppWeb in addition.
[0324/204643.536208:WARNING:runtime_features.cc(728)] AttributionReportingCrossAppWeb cannot be enabled in this configuration. Use --enable-features=ConversionMeasurement,AttributionReportingCrossAppWeb in addition.
486905 bytes written to file /var/folders/wf/rwynm7692pdct5_h_qxs3q180000gq/T/tmphwnootzj/temp.png


In [223]:
len(letty_options)

14

In [224]:
letty_options[0]

[{'(1) Houston',
  '(1) North Carolina',
  '(1) Purdue',
  '(1) Uconn',
  '(11) Duquesne',
  '(11) NC State',
  '(11) Oregon',
  '(12) Grand Canyon',
  '(12) James Madison',
  '(13) Yale',
  '(14) Oakland',
  '(2) Arizona',
  '(2) Iowa St.',
  '(2) Marquette',
  '(2) Tennessee',
  '(3) Baylor',
  '(3) Creighton',
  '(3) Illinois',
  '(4) Alabama',
  '(4) Duke',
  '(4) Kansas',
  '(5) Gonzaga',
  '(5) San Diego St',
  '(6) Clemson',
  '(7) Dayton',
  '(7) Texas',
  '(7) Washington St.',
  '(8) Utah St.',
  '(9) Michigan St.',
  '(9) Northwestern',
  '(9) Texas A&M',
  '**(10) Colorado;Boise St.'},
 {'(1) Houston',
  '(1) North Carolina',
  '(1) Purdue',
  '(1) Uconn',
  '(11) NC State',
  '(2) Arizona',
  '(2) Iowa St.',
  '(2) Marquette',
  '(2) Tennessee',
  '(3) Creighton',
  '(3) Illinois',
  '(4) Alabama',
  '(4) Duke',
  '(5) Gonzaga',
  '(5) San Diego St',
  '(6) Clemson'},
 {'(1) Houston',
  '(1) North Carolina',
  '(2) Marquette',
  '(3) Creighton',
  '(3) Illinois',
  '(5) Gon